# Data pipeline: train / validation / test datasets

Builds training examples and labels for the FCN3 → hourly-wind refinement model (GitHub issue #1).
See `data/data.md` for a description of the raw data and the reasoning behind the split.

**Task (one *window*):** given ERA5 at `t0` and the FCN3 ensemble at `t0+6h`, predict ERA5 u/v at `t0+1h … t0+6h`.

The pipeline has two stages:

1. **Extract windows** (`extract_all`): slow, run once. For every month in `data/fcn3_region/`, it pulls out the
   pieces of each init that a phase-1 window needs and saves them, full-domain, to
   `data/processed/windows/YYYY-MM.nc`. It doesn't depend on the split, and months already extracted are skipped.
2. **Make examples** (`make_patch_dataset`): fast, run as often as you like. It assigns each window to
   train/val/test using `SPLIT_RULES`, then cuts `window_size × window_size` patches into arrays `X`, `Y`.
   `window_size=1` is the pointwise model.

**Adding data later:** drop new `fcn3_YYYY-MM.nc` + `era5_hourly_YYYY-MM.nc` pairs (same format) into
`data/fcn3_region/` and re-run the notebook. New months are extracted and assigned to splits by `SPLIT_RULES`.
Normalization stats are recomputed automatically when the training set changes.

In [1]:
from pathlib import Path
import hashlib, json, re
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "CLAUDE.md").exists():
            return p
    raise FileNotFoundError("Run this notebook from inside the repo")

REPO = find_repo_root()
RAW_DIR = REPO / "data" / "fcn3_region"
OUT_DIR = REPO / "data" / "processed"
WINDOW_DIR = OUT_DIR / "windows"
STATS_PATH = OUT_DIR / "stats.nc"

HOURS = [1, 2, 3, 4, 5, 6]          # target hours after t0
WINDOW_LEAD = 6                     # FCN3 lead used as the window's far endpoint
FCN3_TRAIN_END = pd.Timestamp("2016-12-31T23:59")  # FCN3 trained on ERA5 1980-2016 (arXiv:2507.12144)
print(REPO)

/home/hocke/Balloon_Trajectory_Prediction


## 1. Discover raw months
A month is usable when both its FCN3 file and its hourly-ERA5 file exist.

In [2]:
def discover_months(raw_dir=RAW_DIR):
    # Return sorted 'YYYY-MM' strings that have both an FCN3 and an ERA5-hourly file.
    fcn3 = {re.search(r"fcn3_(\d{4}-\d{2})\.nc$", p.name).group(1) for p in raw_dir.glob("fcn3_????-??.nc")}
    era5 = {re.search(r"era5_hourly_(\d{4}-\d{2})\.nc$", p.name).group(1) for p in raw_dir.glob("era5_hourly_????-??.nc")}
    missing = fcn3 ^ era5
    if missing:
        print("Skipping months without both files:", sorted(missing))
    return sorted(fcn3 & era5)

MONTHS = discover_months()
MONTHS

['2020-01', '2020-07', '2021-01', '2021-07']

## 2. Split rules
Splits are assigned **per window by its init time**, never per patch, so every patch from one window lands in the
same split. Rules (team decision, see `data/data.md`):

- **test:** any init in `test_years`. These must be after FCN3's training period (1980–2016).
- **train only:** inits inside FCN3's training period. Training there is fine, since we learn finer time
  resolution, but validation should look like test.
- **val:** inits on `val_days` of every other month. **buffer:** `buffer_days` are dropped, so no truth hour is
  shared between train and val.
- **train:** everything else.

When more data is added, edit `test_years` here and nothing else.

In [3]:
SPLIT_RULES = {
    "test_years": [2021],
    "val_days": list(range(26, 32)),
    "buffer_days": [24, 25],
}

def assign_split(init, rules=SPLIT_RULES):
    # Return 'train', 'val', 'test' or None (buffer, dropped) for one init time.
    init = pd.Timestamp(init)
    if init.year in rules["test_years"]:
        if init <= FCN3_TRAIN_END:
            raise ValueError(f"test year {init.year} overlaps FCN3's training data (<= 2016)")
        return "test"
    if init <= FCN3_TRAIN_END:
        return "train"
    if init.day in rules["buffer_days"]:
        return None
    if init.day in rules["val_days"]:
        return "val"
    return "train"

## 3. Extract windows (stage 1, cached)
Each saved window contains, on the full 201×241 grid:

| variable | dims | source |
|---|---|---|
| `t0_uv` | (init, var=u/v, level=10, lat, lon) | ERA5 at t0, all 10 levels. This is real input: FCN3 starts from ERA5, so it is known at forecast time |
| `t0_zt` | (init, var=z/t, level=5, lat, lon) | FCN3 lead 0 (= ERA5) z, t |
| `t6` | (init, member, var=u/v/z/t, level=5, lat, lon) | FCN3 ensemble at lead 6h |
| `y` | (init, hour=1..6, var=u/v, level=10, lat, lon) | ERA5 truth at t0+1h … t0+6h |

Inits whose truth hours are missing from the ERA5 file are skipped. Data is written one init at a time, so memory
use stays around 50 MB.

In [4]:
def extract_month(month, overwrite=False):
    out = WINDOW_DIR / f"{month}.nc"
    if out.exists() and not overwrite:
        return out
    WINDOW_DIR.mkdir(parents=True, exist_ok=True)
    # Files are always closed explicitly: netCDF4/HDF5 can segfault when the garbage collector closes a
    # forgotten file in the middle of another HDF5 call.
    with xr.open_dataset(RAW_DIR / f"fcn3_{month}.nc") as f, xr.open_dataset(RAW_DIR / f"era5_hourly_{month}.nc") as e:
        n = _write_windows(f.fcn3, e.era5, month, out)
    print(f"{month}: wrote {n} windows -> {out.relative_to(REPO)}")
    return out

def _write_windows(fcn3, era5, month, out):
    if not (np.array_equal(fcn3.lat, era5.lat) and np.array_equal(fcn3.lon, era5.lon)):
        raise ValueError(f"{month}: FCN3 and ERA5 grids differ")

    era5_times = set(pd.to_datetime(era5.time.values))
    inits = [pd.Timestamp(t) for t in fcn3.init.values
             if all(pd.Timestamp(t) + pd.Timedelta(h, "h") in era5_times for h in [0, *HOURS])]
    shared = [int(l) for l in fcn3.level.values]
    lat, lon = fcn3.lat.values, fcn3.lon.values

    tmp = out.with_suffix(".nc.partial")
    with netCDF4.Dataset(tmp, "w") as nc:
        dims = dict(init=None, member=fcn3.sizes["member"], hour=len(HOURS), uv=2, zt=2, var4=4,
                    level=era5.sizes["level"], level5=len(shared), lat=len(lat), lon=len(lon))
        for d, n in dims.items():
            nc.createDimension(d, n)
        nc.createVariable("init", "f8", ("init",)).units = "hours since 1970-01-01"
        for name, vals in [("lat", lat), ("lon", lon), ("level", era5.level.values), ("level5", shared),
                           ("hour", HOURS), ("member", fcn3.member.values)]:
            nc.createVariable(name, vals.dtype if hasattr(vals, "dtype") else "i8", (name,))[:] = vals
        def var(name, dims):
            chunks = [1] + [dims_n for dims_n in (nc.dimensions[d].size for d in dims[1:])]
            return nc.createVariable(name, "f4", dims, zlib=True, complevel=1, chunksizes=chunks)
        v_t0uv = var("t0_uv", ("init", "uv", "level", "lat", "lon"))
        v_t0zt = var("t0_zt", ("init", "zt", "level5", "lat", "lon"))
        v_t6 = var("t6", ("init", "member", "var4", "level5", "lat", "lon"))
        v_y = var("y", ("init", "hour", "uv", "level", "lat", "lon"))
        nc.setncatts(dict(month=month, var4="u,v,z,t", uv="u,v", zt="z,t",
                          description="Phase-1 windows: ERA5 t0 + FCN3 lead-6 ensemble -> ERA5 t0+1..6h"))

        for k, t in enumerate(inits):
            fc = fcn3.sel(init=t)
            nc["init"][k] = (t - pd.Timestamp("1970-01-01")) / pd.Timedelta(1, "h")
            v_t0uv[k] = era5.sel(time=t).values
            v_t0zt[k] = fc.sel(lead=0, var=["z", "t"]).isel(member=0).values
            v_t6[k] = fc.sel(lead=WINDOW_LEAD, var=["u", "v", "z", "t"]).values
            v_y[k] = era5.sel(time=[t + pd.Timedelta(h, "h") for h in HOURS]).values
    tmp.rename(out)
    return len(inits)

def extract_all(months=None, overwrite=False):
    return [extract_month(m, overwrite) for m in (months or discover_months())]

def open_windows(month):
    # Use as `with open_windows(m) as ds:` so the file is closed deterministically (see extract_month).
    return xr.open_dataset(WINDOW_DIR / f"{month}.nc")  # init decodes to datetime64 via its CF units

In [5]:
%%time
extract_all()

CPU times: user 742 μs, sys: 80 μs, total: 822 μs
Wall time: 576 μs


[PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2020-01.nc'),
 PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2020-07.nc'),
 PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2021-01.nc'),
 PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2021-07.nc')]

### Split summary

In [6]:
def window_index(rules=SPLIT_RULES, months=None):
    # One row per extracted window: month, init, split.
    rows = []
    for m in months or discover_months():
        with open_windows(m) as ds:
            inits = ds.init.values
        for t in inits:
            rows.append(dict(month=m, init=pd.Timestamp(t), split=assign_split(t, rules)))
    return pd.DataFrame(rows)

index = window_index()
index.fillna({"split": "buffer"}).pivot_table(index="month", columns="split", values="init",
                                              aggfunc="count", fill_value=0, margins=True)

split,buffer,test,train,val,All
month,,,,,
2020-01,4,0,46,12,62
2020-07,4,0,46,12,62
2021-01,0,62,0,0,62
2021-07,0,62,0,0,62
All,8,124,92,24,248


## 4. Normalization statistics (train split only)
Each physical variable/level (`u50`, `z100`, …) gets one mean/std pair, computed from the **train** windows' t0
fields. The same pair is applied to t0, the t6 members and the targets, so every field shares one scale. That keeps
residual-style models simple: target minus input stays meaningful.

Stats are cached in `stats.nc` together with a hash of the train inits, and recomputed when the training set changes.

In [7]:
def _train_signature(index):
    train = index[index.split == "train"].init.astype(str).tolist()
    return hashlib.sha1(json.dumps(train).encode()).hexdigest(), len(train)

def load_stats(index=None, rules=SPLIT_RULES):
    index = window_index(rules) if index is None else index
    sig, n = _train_signature(index)
    if STATS_PATH.exists():
        with xr.open_dataset(STATS_PATH) as f:
            stats = f.load()
        if stats.attrs.get("train_signature") == sig:
            return stats
    sums, sq, count = {}, {}, 0
    for m, group in index[index.split == "train"].groupby("month"):
        with open_windows(m) as ds:
            for t in group.init.values:
                w = ds.sel(init=t)
                for name, arr in _named_fields(w["t0_uv"].values, w["t0_zt"].values, ds).items():
                    sums[name] = sums.get(name, 0.0) + arr.sum(dtype=np.float64)
                    sq[name] = sq.get(name, 0.0) + (arr.astype(np.float64) ** 2).sum()
                count += arr.size
    names = list(sums)
    mean = np.array([sums[k] / count for k in names])
    std = np.sqrt(np.array([sq[k] / count for k in names]) - mean ** 2)
    stats = xr.Dataset({"mean": ("field", mean), "std": ("field", std)}, coords={"field": names},
                       attrs=dict(train_signature=sig, n_train_windows=n))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    stats.to_netcdf(STATS_PATH)
    print(f"computed stats from {n} train windows")
    return stats

def _named_fields(t0_uv, t0_zt, ds):
    # Map 'u50', 'v30', 'z100', ... -> 2D field for one window's t0.
    out = {}
    for i, v in enumerate("uv"):
        for j, lev in enumerate(ds.level.values):
            out[f"{v}{int(lev)}"] = t0_uv[i, j]
    for i, v in enumerate("zt"):
        for j, lev in enumerate(ds.level5.values):
            out[f"{v}{int(lev)}"] = t0_zt[i, j]
    return out

stats = load_stats(index)
stats.to_dataframe().round(2).T

field,u30,u50,u70,u100,u125,u150,u175,u200,u225,u250,...,z50,z100,z150,z200,z250,t50,t100,t150,t200,t250
mean,-1.65,-3.01,0.52,6.83,12.44,14.81,15.33,14.79,13.78,12.66,...,202295.56,162277.51,138655.14,120988.15,106599.57,207.26,198.41,208.76,219.56,229.97
std,12.42,12.38,12.00,14.07,15.38,17.78,19.50,20.33,20.26,19.59,...,679.48,1257.61,1792.33,1901.62,1780.64,3.56,6.78,4.24,2.69,3.77


## 5. Make examples (stage 2)

`make_patch_dataset(split, window_size, ...)` returns a dict:

| key | shape | meaning |
|---|---|---|
| `X` | (N, C, window_size, window_size) | input channels, listed in `x_names` |
| `Y` | (N, 6, 2, L, label_size, label_size) | targets: hour 1–6 × (u, v) × `target_levels` |
| `meta` | DataFrame, N rows | `month, init, split, i, j`: the window's top-left grid index, for stitching/analysis |
| `x_names`, `target_levels`, `label_offset` | | bookkeeping |

**Input channels** (`C` = 20 + 10 + t6 + 3):
- `t0_u{lev}`, `t0_v{lev}`: ERA5 t0 at all 10 levels (20)
- `t0_z{lev}`, `t0_t{lev}`: t0 at the 5 FCN3 levels (10)
- t6, depending on `member_mode`:
  - `"stack"`: every member as its own channels, `t6_m{k}_{var}{lev}` (4×20=80). The model sees member order,
    which is arbitrary.
  - `"meanstd"`: `t6_mean_*`, `t6_std_*` (40). Permutation-invariant, and works if member counts differ between
    months.
  - `"mean"`: ensemble mean only (20)
- `sin_lat`, `sin_lon`, `cos_lon`: absolute position, since a patch doesn't otherwise know where it is (3)

**Choosing patches:**
- `window_size`: input patch side. **1 = pointwise.**
- `label_size`: the side of the central region that gets labels (default = `window_size`). Setting it smaller
  gives a context halo. For example, `window_size=64, label_size=48` gives an 8-cell border of input-only context.
  `window_size − label_size` must be even.
- Pass **either** `n_per_window` (random positions, for training; seeded) **or** `stride` (a regular grid, for
  evaluation). With `stride=label_size` the label regions tile the domain without gaps. The last row/column is
  shifted to touch the edge, and with a halo the outer `(window_size−label_size)/2` cells of the domain are never
  labelled.

Normalized values: `x_norm = (x − mean) / std` per physical field. `t6_std_*` channels are divided by std only.
Use `denormalize_y` to get m/s back.

In [8]:
def _stat(stats, name):
    return float(stats["mean"].sel(field=name)), float(stats["std"].sel(field=name))

def _window_fields(w, levels, levels5, member_mode, stats):
    # Build (C, H, W) input and (6, 2, 10, H, W) target for one window, with channel names.
    chans, names = [], []
    def add(arr, name, field, center=True):
        if stats is not None:
            mu, sd = _stat(stats, field)
            arr = (arr - mu) / sd if center else arr / sd
        chans.append(arr); names.append(name)
    t0_uv, t0_zt, t6 = w["t0_uv"].values, w["t0_zt"].values, w["t6"].values
    for i, v in enumerate("uv"):
        for j, lev in enumerate(levels):
            add(t0_uv[i, j], f"t0_{v}{lev}", f"{v}{lev}")
    for i, v in enumerate("zt"):
        for j, lev in enumerate(levels5):
            add(t0_zt[i, j], f"t0_{v}{lev}", f"{v}{lev}")
    if member_mode == "stack":
        groups = [(f"m{k}", t6[k], True) for k in range(t6.shape[0])]
    elif member_mode == "meanstd":
        groups = [("mean", t6.mean(0), True), ("std", t6.std(0, ddof=1), False)]
    elif member_mode == "mean":
        groups = [("mean", t6.mean(0), True)]
    else:
        raise ValueError(f"member_mode must be 'stack', 'meanstd' or 'mean', got {member_mode!r}")
    for tag, arr, center in groups:
        for i, v in enumerate("uvzt"):
            for j, lev in enumerate(levels5):
                add(arr[i, j], f"t6_{tag}_{v}{lev}", f"{v}{lev}", center)
    lat = np.deg2rad(w.lat.values)[:, None] * np.ones((1, w.sizes["lon"]))
    lon = np.deg2rad(w.lon.values)[None, :] * np.ones((w.sizes["lat"], 1))
    chans += [np.sin(lat), np.sin(lon), np.cos(lon)]; names += ["sin_lat", "sin_lon", "cos_lon"]

    y = w["y"].values.copy()
    if stats is not None:
        for i, v in enumerate("uv"):
            for j, lev in enumerate(levels):
                mu, sd = _stat(stats, f"{v}{lev}")
                y[:, i, j] = (y[:, i, j] - mu) / sd
    return np.stack(chans).astype(np.float32), y.astype(np.float32), names

def _positions(n_rows, n_cols, k, n_per_window, stride, rng):
    if (n_per_window is None) == (stride is None):
        raise ValueError("pass exactly one of n_per_window (random) or stride (grid)")
    if n_per_window is not None:
        return rng.integers(0, n_rows - k + 1, n_per_window), rng.integers(0, n_cols - k + 1, n_per_window)
    rows = sorted(set(range(0, n_rows - k + 1, stride)) | {n_rows - k})
    cols = sorted(set(range(0, n_cols - k + 1, stride)) | {n_cols - k})
    I, J = np.meshgrid(rows, cols, indexing="ij")
    return I.ravel(), J.ravel()

def make_patch_dataset(split, window_size, label_size=None, n_per_window=None, stride=None,
                       member_mode="stack", target_levels=None, normalize=True, seed=0,
                       rules=SPLIT_RULES, max_gb=6.0):
    k = window_size
    l = k if label_size is None else label_size
    if l > k or (k - l) % 2:
        raise ValueError("label_size must be <= window_size with an even difference")
    off = (k - l) // 2
    index = window_index(rules)
    windows = index[index.split == split]
    if windows.empty:
        raise ValueError(f"no windows in split {split!r}")
    stats = load_stats(index, rules) if normalize else None
    rng = np.random.default_rng(seed)

    with open_windows(windows.month.iloc[0]) as first:
        levels = [int(v) for v in first.level.values]
        levels5 = [int(v) for v in first.level5.values]
        n_members = first.sizes["member"]
        n_rows, n_cols = first.sizes["lat"], first.sizes["lon"]
    target_levels = levels if target_levels is None else list(target_levels)
    lev_idx = [levels.index(v) for v in target_levels]
    n_pos = len(_positions(n_rows, n_cols, k, n_per_window, stride, rng)[0])
    n_chan = 30 + {"stack": 20 * n_members, "meanstd": 40, "mean": 20}[member_mode] + 3
    N = len(windows) * n_pos
    gb = N * 4 * (n_chan * k * k + len(HOURS) * 2 * len(target_levels) * l * l) / 1e9
    if gb > max_gb:
        raise MemoryError(f"dataset would be {gb:.1f} GB (> max_gb={max_gb}); use fewer/smaller patches")

    X = np.empty((N, n_chan, k, k), np.float32)
    Y = np.empty((N, len(HOURS), 2, len(target_levels), l, l), np.float32)
    meta, n = [], 0
    for month, group in windows.groupby("month", sort=True):
        with open_windows(month) as ds:
            if member_mode == "stack" and ds.sizes["member"] != n_members:
                raise ValueError(f"{month} has {ds.sizes['member']} members, expected {n_members}; "
                                 "use member_mode='meanstd' or 'mean' to mix months")
            for t in group.init.values:
                x_full, y_full, x_names = _window_fields(ds.sel(init=t).load(), levels, levels5, member_mode, stats)
                I, J = _positions(n_rows, n_cols, k, n_per_window, stride, rng)
                xw = np.lib.stride_tricks.sliding_window_view(x_full, (k, k), axis=(1, 2))
                X[n:n + len(I)] = xw[:, I, J].transpose(1, 0, 2, 3)
                yw = np.lib.stride_tricks.sliding_window_view(y_full[:, :, lev_idx], (l, l), axis=(3, 4))
                Y[n:n + len(I)] = yw[:, :, :, I + off, J + off].transpose(3, 0, 1, 2, 4, 5)
                meta += [dict(month=month, init=pd.Timestamp(t), split=split, i=i, j=j) for i, j in zip(I, J)]
                n += len(I)
    print(f"{split}: {len(windows)} windows x {n_pos} patches = {N} examples, X {X.shape}, Y {Y.shape}, {gb:.2f} GB")
    return dict(X=X, Y=Y, meta=pd.DataFrame(meta), x_names=x_names, target_levels=target_levels,
                label_offset=off, normalized=normalize)

def denormalize_y(Y, target_levels, stats=None):
    # Normalized targets (N, 6, 2, L, h, w) -> m/s.
    stats = load_stats() if stats is None else stats
    out = Y.copy()
    for i, v in enumerate("uv"):
        for j, lev in enumerate(target_levels):
            mu, sd = _stat(stats, f"{v}{lev}")
            out[:, :, i, j] = out[:, :, i, j] * sd + mu
    return out

## 6. Examples

In [9]:
# Pointwise (window_size=1): 2,000 random grid points per training window
point = make_patch_dataset("train", window_size=1, n_per_window=2000)
X_point = point["X"][:, :, 0, 0]          # (N, C) for an MLP
Y_point = point["Y"][..., 0, 0]           # (N, 6, 2, 10)
X_point.shape, Y_point.shape

train: 92 windows x 2000 patches = 184000 examples, X (184000, 113, 1, 1), Y (184000, 6, 2, 10, 1, 1), 0.17 GB


((184000, 113), (184000, 6, 2, 10))

In [10]:
# Patches for training: 64x64 inputs, labels on the central 48x48, ensemble summarized as mean/std
train = make_patch_dataset("train", window_size=64, label_size=48, n_per_window=8, member_mode="meanstd")
train["X"].shape, train["Y"].shape, train["meta"].head(3)

train: 92 windows x 8 patches = 736 examples, X (736, 73, 64, 64), Y (736, 6, 2, 10, 48, 48), 1.69 GB


((736, 73, 64, 64),
 (736, 6, 2, 10, 48, 48),
      month       init  split   i    j
 0  2020-01 2020-01-01  train  87   70
 1  2020-01 2020-01-01  train  75  152
 2  2020-01 2020-01-01  train  77   98)

In [11]:
# Evaluation: tile the full domain (label regions tile exactly with stride = label_size)
val = make_patch_dataset("val", window_size=64, label_size=48, stride=48, member_mode="meanstd")
val["meta"].groupby("init").size().head(3)

val: 24 windows x 20 patches = 480 examples, X (480, 73, 64, 64), Y (480, 6, 2, 10, 48, 48), 1.10 GB


init
2020-01-26 00:00:00    20
2020-01-26 12:00:00    20
2020-01-27 00:00:00    20
dtype: int64

## 7. Sanity checks
1. Re-derive the **linear-interpolation baseline** from a dataset (unnormalized, pointwise, the 5 shared levels).
   It should match `data/data.md` (≈ 0.8 / 1.25 / 1.4 / 1.4 / 1.3 / 1.3 m/s for hours 1–6). This confirms the
   targets line up with the right hours.
2. Normalize → denormalize round trip.

In [12]:
shared = [50, 100, 150, 200, 250]
chk = make_patch_dataset("val", window_size=1, n_per_window=3000, member_mode="mean",
                         target_levels=shared, normalize=False)
names = chk["x_names"]
x0 = np.stack([chk["X"][:, names.index(f"t0_{v}{lev}"), 0, 0] for v in "uv" for lev in shared], 1)
x6 = np.stack([chk["X"][:, names.index(f"t6_mean_{v}{lev}"), 0, 0] for v in "uv" for lev in shared], 1)
y = chk["Y"][..., 0, 0].reshape(len(x0), 6, -1)
rmse = {h: float(np.sqrt(((x0 + (x6 - x0) * h / 6 - y[:, h - 1]) ** 2).mean())) for h in HOURS}
print("linear interp (ERA5 t0 -> FCN3 mean t6) RMSE by hour:", {h: round(r, 3) for h, r in rmse.items()})
assert 0.5 < rmse[1] < 1.1 and 1.0 < rmse[6] < 1.7, "targets look misaligned"

norm = make_patch_dataset("val", window_size=1, n_per_window=3000, member_mode="mean", target_levels=shared)
assert np.allclose(denormalize_y(norm["Y"], shared), chk["Y"], atol=1e-3)
print("round trip OK")

val: 24 windows x 3000 patches = 72000 examples, X (72000, 53, 1, 1), Y (72000, 6, 2, 5, 1, 1), 0.03 GB
linear interp (ERA5 t0 -> FCN3 mean t6) RMSE by hour: {1: 0.765, 2: 1.208, 3: 1.389, 4: 1.378, 5: 1.285, 6: 1.302}
val: 24 windows x 3000 patches = 72000 examples, X (72000, 53, 1, 1), Y (72000, 6, 2, 5, 1, 1), 0.03 GB
round trip OK
